In [1]:
from neo4j import GraphDatabase
import json
from pathlib import Path
from tqdm.auto import tqdm

PROJECT_ROOT  = Path.home() / "xai-knowledge-graph"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

def load_creds(path=str(Path.home() / "aura_cred.txt")):
    creds = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                creds[k.strip()] = v.strip()
    return creds

creds  = load_creds()
driver = GraphDatabase.driver(
    creds["NEO4J_URI"],
    auth=(creds["NEO4J_USERNAME"], creds["NEO4J_PASSWORD"])
)
DB = creds["NEO4J_DATABASE"]

with driver.session(database=DB) as s:
    print(s.run("RETURN 'connected' AS status").single()["status"])

connected


In [2]:
query ="""
MATCH(p:Paper)
RETURN p.title AS title,
       p. year AS year,
       p.citation_count AS citations
ORDER BY citations DESC
LIMIT 10
"""
print("=== Top 10 Most-Cited XAI Papers ===")
with driver.session(database=DB)as s:
    for row in s.run(query):
        print(f"[{row['citations']:>6}] ({row['year']}) {row['title'][:80]}")
    

=== Top 10 Most-Cited XAI Papers ===
[ 26975] (2016) Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization
[  5264] (2017) Explanation in Artificial Intelligence: Insights from the Social Sciences
[  3132] (2017) Grad-CAM++: Improved Visual Explanations for Deep Convolutional Networks
[  1817] (2018) Consistent Individualized Feature Attribution for Tree Ensembles
[  1089] (2019) Fooling LIME and SHAP: Adversarial Attacks on Post hoc Explanation Methods
[  1012] (2021) Explainable artificial intelligence (XAI) in deep learning-based medical image a
[   936] (2020) Questioning the AI: Informing Design Practices for Explainable AI User Experienc
[   908] (2018) Visual Interpretability for Deep Learning: a Survey
[   749] (2020) Opportunities and Challenges in Explainable Artificial Intelligence (XAI): A Sur
[   671] (2022) From Anecdotal Evidence to Quantitative Evaluation Methods: A Systematic Review 


In [3]:
query="""
MATCH (a: Author)-[:AUTHORED]->(p:Paper)
RETURN 
    a.name AS author,
    count(p) AS paper_count
ORDER BY paper_count DESC
LIMIT 10
"""
print("=== Top 10 most prolific authors in our corpus===")
print(f"{'Author':<22}{'Count':>13}")
with driver.session(database=DB) as s:
    for row in s.run(query):
        print(f"{row['author']:<22} {row['paper_count']:>10}")

=== Top 10 most prolific authors in our corpus===
Author                        Count
S. Lapuschkin                  23
W. Samek                       17
Barbara Hammer                 16
David Martens                  14
Xuanxiang Huang                12
João Marques-Silva             12
P. Biecek                      12
Francesca Toni                 11
Wojciech Samek                 11
Inga Strümke                   11


In [4]:
query="""
MATCH (p:Paper)-[:ABOUT]->(t:Topic)
RETURN
     t.name AS Topic,
     count(p) AS paper_count
ORDER BY paper_count DESC
"""
print("=== Topic distribution ===")
print(f"{'Topic':<22}{'Count':>11}")
with driver.session(database=DB) as s:
    for row in s.run(query):
        print(f"{row['Topic']:<22} {row['paper_count']:>10}")

=== Topic distribution ===
Topic                       Count
Explainability               2240
Interpretability             1817
SHAP                         1057
Healthcare                    903
Transparency                  582
NLP                           483
Feature Attribution           476
Computer Vision               442
Counterfactual                405
Grad-CAM                      395
LIME                          364
Saliency                      330
Post-hoc                      298
Trust                         272
Model-Agnostic                232
Integrated Gradients          167
Attention                     153
Time Series                   151
Fairness                      121
Tabular                       116
Graph Neural Networks          89
Reinforcement                  88
Causal                         55
Federated                      42
Adversarial                    35
Prototype                      23
Concept Activation             22
Rule-based           

In [5]:
query="""
MATCH(a1:Author)-[:AUTHORED]->(p:Paper)<-[:AUTHORED]-(a2:Author)
WHERE a1.name <a2.name
RETURN
      a1.name AS Author_1,
      a2.name AS Author_2,
      count (DISTINCT p) AS Collaboration
ORDER BY Collaboration DESC
LIMIT 10
"""
print("=== Top 10 co-authorship pairs ===")
print(f"{'Author_1':<25} {'Author_2':<25} {'Count':>5}")
with driver.session(database=DB) as s:
    for row in s.run(query):
        print(f"{row['Author_1']:<25} {row['Author_2']:<25} {row['Collaboration']:>5}")

=== Top 10 co-authorship pairs ===
Author_1                  Author_2                  Count
S. Lapuschkin             W. Samek                     11
S. Lapuschkin             Wojciech Samek                9
Fabian Fumagalli          Maximilian Muschalik          9
João Marques-Silva        Xuanxiang Huang               9
Benedict Clark            Rick Wilming                  8
E. Pishgar                M. Pishgar                    8
K. Alaei                  M. Pishgar                    8
Grzegorz J. Nalepa        Szymon Bobek                  8
Antonio Rago              Francesca Toni                8
E. Pishgar                K. Alaei                      8


In [6]:
query="""
MATCH(p:Paper)<-[c:CITES]-(:Paper)
RETURN
    p.title AS title,
    p.year AS year,
    count(c) AS in_corpus_citations
ORDER BY in_corpus_citations DESC
LIMIT 10
"""

print("=== Most influential papers within our corpus  ===")
with driver.session(database=DB)as s: 
    for row in s.run(query):
        print(f"[{row['in_corpus_citations']:>6}] ({row['year']}) {row['title'][:80]}")
    

=== Most influential papers within our corpus  ===
[   387] (2016) Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization
[   168] (2017) Explanation in Artificial Intelligence: Insights from the Social Sciences
[    85] (2017) Grad-CAM++: Improved Visual Explanations for Deep Convolutional Networks
[    65] (2019) Fooling LIME and SHAP: Adversarial Attacks on Post hoc Explanation Methods
[    43] (2022) From Anecdotal Evidence to Quantitative Evaluation Methods: A Systematic Review 
[    43] (2020) Opportunities and Challenges in Explainable Artificial Intelligence (XAI): A Sur
[    42] (2018) Consistent Individualized Feature Attribution for Tree Ensembles
[    26] (2020) Questioning the AI: Informing Design Practices for Explainable AI User Experienc
[    23] (2022) From Attribution Maps to Human-Understandable Explanations through Concept Relev
[    22] (2020) Evaluating Explainable AI: Which Algorithmic Explanations Help Users Predict Mod


In [7]:
query="""
MATCH(p:Paper)-[:ABOUT]->(:Topic {name: "Interpretability"})
MATCH(p:Paper)-[:ABOUT]->(:Topic {name: "Computer Vision"})
RETURN
    p.title AS title,
    p.year AS year,
    p.citation_count AS citation_count
ORDER BY citation_count DESC
LIMIT 10
"""

print("=== Bridge papers — papers that connect Interpretability and Computer Vision  ===")
print(f"{'Title':<80} {'Year':>6} {'Citations':>10}")
print("-" * 100)
with driver.session(database=DB)as s:
    for row in s.run(query):
        print(f"{row['title'][:80]:<80} {row['year']:>6} {row['citation_count']:>10,}")

=== Bridge papers — papers that connect Interpretability and Computer Vision  ===
Title                                                                              Year  Citations
----------------------------------------------------------------------------------------------------
Grad-CAM++: Improved Visual Explanations for Deep Convolutional Networks           2017      3,132
Visual Interpretability for Deep Learning: a Survey                                2018        908
Towards Interpretable Semantic Segmentation via Gradient-weighted Class Activati   2020        196
Explainable Deep Learning Methods in Medical Image Classification: A Survey        2022        139
Explainable Artificial Intelligence for Human Decision-Support System in Medical   2021        136
Improving Deep Learning Interpretability by Saliency Guided Training               2021        115
Fault Diagnosis using eXplainable AI: a Transfer Learning-based Approach for Rot   2022         86
Is Grad-CAM Explainable i

In [8]:
query="""
MATCH (p:Paper)-[:ABOUT]->(t1:Topic),
      (p)-[:ABOUT]->(t2:Topic)
WHERE (t1.name)<(t2.name)
RETURN
    t1.name AS Topic1,
    t2.name AS Topic2,
    count(distinct p) AS Count
ORDER BY Count DESC
LIMIT 15
"""
print("=== Top 15 Topic co-occurrence ===")
print(f"{'Topic1':<25} {'Topic2':<25} {'Count':>5}")
with driver.session(database=DB) as s:
    for row in s.run(query):
        print(f"{row['Topic1']:<25} {row['Topic2']:<25} {row['Count']:>5}")


=== Top 15 Topic co-occurrence ===
Topic1                    Topic2                    Count
Explainability            Interpretability            909
Explainability            SHAP                        558
Explainability            Healthcare                  526
Healthcare                Interpretability            524
Interpretability          SHAP                        507
Explainability            Transparency                472
Interpretability          Transparency                337
Healthcare                SHAP                        252
Explainability            Feature Attribution         251
Computer Vision           Explainability              248
Explainability            LIME                        240
Interpretability          NLP                         238
Explainability            NLP                         237
Computer Vision           Interpretability            230
Explainability            Trust                       223


In [9]:
query="""
MATCH (a:Author)-[:AUTHORED]->(p:Paper)-[:ABOUT]->(:Topic {name: "SHAP"}),
      (p)<-[c:CITES]-(:Paper)
RETURN
    a.name AS Author,
    count(DISTINCT p) AS shap_papers,
    count(DISTINCT c) AS in_corpus_citations
ORDER BY in_corpus_citations DESC
LIMIT 10
"""
print("=== Top 10 authors most cited within a specific topic area ===")
print(f"{'Author':<30} {'In-Corpus Cites':>16} {'SHAP Papers':>13}")
print("-" * 62)
with driver.session(database=DB) as s:
    for row in s.run(query):
        print(f"{row['Author']:<30} {row['in_corpus_citations']:>16} {row['shap_papers']:>13}")

=== Top 10 authors most cited within a specific topic area ===
Author                          In-Corpus Cites   SHAP Papers
--------------------------------------------------------------
Emily Jia                                    65             1
Sophie Hilgard                               65             1
Himabindu Lakkaraju                          65             1
Dylan Slack                                  65             1
Sameer Singh                                 65             1
Su-In Lee                                    43             2
Scott M. Lundberg                            43             2
Gabriel Erion-Barner                         42             1
Barbara Hammer                               24             4
Maximilian Muschalik                         24             4


In [10]:
query="""
MATCH (p:Paper)-[:ABOUT]->(t:Topic)

WHERE p.year >= 2023
WITH t,p
ORDER BY t.name, p.citation_count DESC
WITH t, collect(p)[0] AS top_paper

RETURN
    t.name AS Topic,
    top_paper.title AS Title,
    top_paper.year AS Year,
    top_paper.citation_count AS citation_count
    
ORDER BY Topic
"""
print("=== Most influential paper per topic, last 3 years  ===")
print(f"{'Topic':<25} {'Title':<60} {'Year':>6} {'Citations':>10}")
print("-" * 100)
with driver.session(database=DB)as s:
    for row in s.run(query):
        print(f"{row['Topic'][:80]:<25} {row['Title'][:60]:<60} {row['Year']:>6} {row['citation_count']:>10,}")


=== Most influential paper per topic, last 3 years  ===
Topic                     Title                                                          Year  Citations
----------------------------------------------------------------------------------------------------
Adversarial               Distance-Restricted Explanations: Theoretical Underpinnings    2024         22
Attention                 ConceptAttention: Diffusion Transformers Learn Highly Interp   2025         42
Bias                      Visual-TCAV: Concept-based Attribution and Saliency Maps for   2024         11
Causal                    The role of causality in explainable artificial intelligence   2023         49
Computer Vision           Is Grad-CAM Explainable in Medical Images?                     2023         78
Concept Activation        Visual-TCAV: Concept-based Attribution and Saliency Maps for   2024         11
Counterfactual            The role of causality in explainable artificial intelligence   2023         49
Exp

In [11]:
query="""
MATCH (source:Paper)
WHERE source.title CONTAINS "Grad-CAM"
MATCH (source)-[:ABOUT]->(t:Topic)<-[:ABOUT]-(other:Paper)
WHERE other <> source

WITH
    source,
    other,
    Collect(t.name) AS Shared_topic,
    Count(DISTINCT t) AS Shared_topic_count
RETURN
    source.title AS Source_Paper,
    other.title As Similar_paper,
    Shared_topic,
    Shared_topic_count
ORDER BY Shared_topic_count DESC
LIMIT 10
"""

print("=== Top 10 Papers Most Similar to Grad-CAM (by shared topics) ===")
print(f"{'Source':<30} {'Similar Paper':<50} {'Shared':>8}  Topics")
print("-" * 100)
with driver.session(database=DB) as s:
    for row in s.run(query):
        topics_str = ", ".join(row['Shared_topic'])
        print(f"{row['Source_Paper'][:30]:<30} {row['Similar_paper'][:40]:<50} {row['Shared_topic_count']:>5}  {topics_str}")

=== Top 10 Papers Most Similar to Grad-CAM (by shared topics) ===
Source                         Similar Paper                                        Shared  Topics
----------------------------------------------------------------------------------------------------
Enhanced SegNet with Integrate From Explanations to Architecture: Expla               6  Transparency, Explainability, Trust, Interpretability, Grad-CAM, Healthcare
Enhanced SegNet with Integrate Brain Stroke Detection and Classificatio               6  Transparency, Explainability, Trust, Interpretability, Grad-CAM, Healthcare
Enhanced SegNet with Integrate Choose Your Explanation: A Comparison of               6  Transparency, Explainability, Trust, Interpretability, Grad-CAM, Healthcare
Seeing Isn't Always Believing: Dual-Modal Lung Cancer AI: Interpretable               6  Computer Vision, Explainability, Interpretability, Grad-CAM, Healthcare, Saliency
Enhanced SegNet with Integrate Explainable AI: A Combined XAI Framew

In [14]:
query="""
MATCH path= shortestPath((source:Paper)-[:CITES*]->(target:Paper))
WHERE source.year>=2025 
    AND target.arxiv_id = "1610.02391"
RETURN
    [n IN nodes(path) | n.title] AS titles, 
    length(path) AS hops
ORDER BY hops ASC
LIMIT 10
"""
print("=== Shortest Citation Paths from 2025+ Papers to Grad-CAM ===")
print()
with driver.session(database=DB) as s:
    for row in s.run(query):
        chain = "\n   → ".join(t[:80] for t in row['titles'])
        print(f"[{row['hops']} hops]\n   {chain}")
        print()



=== Shortest Citation Paths from 2025+ Papers to Grad-CAM ===

[1 hops]
   A Two-Stage LLM Framework for Accessible and Verified XAI Explanations
   → Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization

[1 hops]
   Med-CAM: Minimal Evidence for Explaining Medical Decision Making
   → Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization

[1 hops]
   Explainable Fall Detection for Elderly Monitoring via Temporally Stable SHAP in 
   → Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization

[1 hops]
   On the Importance and Evaluation of Narrativity in Natural Language AI Explanati
   → Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization

[1 hops]
   Attention-ResUNet for Automated Fetal Head Segmentation
   → Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization

[1 hops]
   Multimodal Fusion of Histopathology Images and Electronic Health Records for